In [80]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movie_ratings = pd.read_csv("movie_ratings.csv")

In [6]:
# data exploration

In [7]:
movie_ratings

,user_id,movie_id,rating
0,Sam,Lion King,4
1,Sam,Jerry Maguire,4
2,Sam,Roman Holidays,5
3,Sam,Downfall,1
4,Eva,Titanic,2
5,Eva,Jerry Maguire,1
6,Eva,Inception,4
7,Eva,Man on Wire,5
8,Eva,The Social Dilemma,5
9,Pat,Titanic,3


In [12]:
# get statistics of data 
print("Number of ratings:", len(movie_ratings))
print("Average rating:  %0.3f" % (np.mean(movie_ratings["rating"])))
N = len(np.unique(movie_ratings["user_id"]))
M = len(np.unique(movie_ratings["movie_id"]))
print("Number of users (N): %d" % N)
print("Number of items (M): %d" % M)
print("Fraction non-nan ratings: %0.3f" % (len(movie_ratings) / (N * M)))

Number of ratings: 21
Average rating:  3.524
Number of users (N): 4
Number of items (M): 12
Fraction non-nan ratings: 0.438


# Construct utility matrix

In [14]:
utility_matrix = movie_ratings.pivot(index="user_id",columns="movie_id",values="rating")

In [15]:
utility_matrix

movie_id,A Beautiful Mind,Bambi,Cast Away,Downfall,Inception,Jerry Maguire,Lion King,Malcolm x,Man on Wire,Roman Holidays,The Social Dilemma,Titanic
user_id,,,,,,,,,,,,
Eva,NaN,NaN,NaN,NaN,4.0,1.0,NaN,NaN,5.0,NaN,5.0,2.0
Jim,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,5.0,NaN,5.0,2.0
Pat,3.0,4.0,3.0,2.0,NaN,5.0,4.0,NaN,NaN,NaN,NaN,3.0
Sam,NaN,NaN,NaN,1.0,NaN,4.0,4.0,NaN,NaN,5.0,NaN,NaN


# user-based collaborative filtering

In [48]:
# Calculate similarity between users.
# Users need at least two commonly rated movies for a correlation.
user_similarity = utility_matrix.T.corr(
    method="pearson",
    min_periods=2
)
user_similarity

user_id,Eva,Jim,Pat,Sam
user_id,,,,
Eva,1.0,1.0,-1.000000,NaN
Jim,1.0,1.0,1.000000,NaN
Pat,-1.0,1.0,1.000000,0.944911
Sam,NaN,NaN,0.944911,1.000000


In [49]:
def predict_rating(user_id, movie_id):
    """Predict a rating using the formula shown."""

    # Users who rated this movie
    movie_ratings = utility_matrix[movie_id].dropna()

    # Similarities between the target user and those users
    similarities = user_similarity.loc[user_id].reindex(movie_ratings.index)
    similarities = similarities.dropna().drop(index=user_id, errors="ignore")

    if similarities.empty:
        return np.nan

    # Each neighbour's rating adjusted by their mean rating
    neighbor_means = utility_matrix.loc[similarities.index].mean(axis=1)
    rating_deviations = movie_ratings.loc[similarities.index] - neighbor_means

    # Apply the formula:
    # target user's mean + weighted deviations / sum of similarities
    denominator = similarities.sum()

    # The prediction is undefined if the denominator is zero
    if np.isclose(denominator, 0):
        return np.nan

    numerator = (similarities * rating_deviations).sum()

    return utility_matrix.loc[user_id].mean() + numerator / denominator

In [50]:
def recommend_movies(user_id, n=3):
    """Return up to n highest-rated recommendations for a user."""

    unseen_movies = utility_matrix.columns[utility_matrix.loc[user_id].isna()]

    predictions = {movie: predict_rating(user_id, movie) for movie in unseen_movies}

    recommendations = pd.Series(predictions).dropna()
    return recommendations.sort_values(ascending=False).head(n)

In [51]:
# Example: recommend movies for Sam
print(recommend_movies("Eva"))

Bambi               3.971429
Malcolm x           3.600000
A Beautiful Mind    2.971429
dtype: float64


# item-based collaborative filtering

In [39]:
# Calculate Pearson similarity between movies.
# NaN ratings are excluded when each pairwise correlation is calculated.
item_similarity = utility_matrix.corr(method="pearson",min_periods=2)

In [40]:
def predict_rating(user_id, movie_id):
    """Predict a user's rating using all positively similar movies they rated."""

    user_ratings = utility_matrix.loc[user_id].dropna()

    similarities = item_similarity[movie_id].reindex(user_ratings.index)
    similarities = similarities.dropna().drop(index=movie_id,errors="ignore")

    # Use all rated movies with positive similarity
    neighbors = similarities[similarities > 0]

    if neighbors.empty:
        return np.nan

    weighted_sum = (neighbors * user_ratings.loc[neighbors.index]).sum()

    return weighted_sum / neighbors.sum()

In [41]:
def recommend_movies(user_id, n=5):
    """Return up to n highest-predicted unrated movies."""

    unseen_movies = utility_matrix.columns[
        utility_matrix.loc[user_id].isna()
    ]

    predictions = {
        movie: predict_rating(user_id, movie)
        for movie in unseen_movies
    }

    return (pd.Series(predictions).dropna().sort_values(ascending=False).head(n))

In [46]:
# Example: recommend movies for Sam
print(recommend_movies("Eva"))

Lion King    2.0
Downfall     1.0
dtype: float64


# Content-based filtering

In [72]:
movies = pd.read_csv("movie_features.csv")
movies

,movie_id,genres,keywords
0,A Beautiful Mind,Romance; Drama,mathematician; genius; Princeton; schizophreni...
1,Bambi,Drama; Children,young deer; forest; friendship; coming of age;...
2,Cast Away,Romance; Drama,stranded island; survival; isolation; castaway...
3,Downfall,Documentary,World War II; Hitler; Berlin bunker; Nazi Germ...
4,Inception,Action; Drama,dreams; heist; subconscious; memory; reality; ...
5,Jerry Maguire,Romance; Drama; Comedy,sports agent; football; career; romance; loyal...
6,Lion King,Drama; Children,lion; savannah; family; responsibility; coming...
7,Malcolm x,Documentary,Malcolm X; civil rights; activism; biography; ...
8,Man on Wire,Documentary,tightrope; high-wire walk; World Trade Center;...
9,Roman Holidays,Romance; Drama; Comedy,princess; Rome; journalist; romance; travel; f...


In [74]:
# Combine genres and keywords into one text description per movie
movies["content"] = (movies["genres"].fillna("") + "; " +movies["keywords"].fillna(""))

# Convert movie descriptions to TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words="english",ngram_range=(1, 2))
movie_features = vectorizer.fit_transform(movies["content"])

movie_ids = movies["movie_id"].tolist()
movie_to_row = {movie_id: row for row, movie_id in enumerate(movie_ids)}


In [93]:
def recommend_movies(user_id, n=3, like_threshold=4):
    """Recommend unrated movies similar to those the user liked."""

    user_ratings = movie_ratings[movie_ratings["user_id"] == user_id]
    
    liked_ids = user_ratings.loc[user_ratings["rating"] >= like_threshold, "movie_id"]
    
    liked_rows = [movie_to_row[movie_id] for movie_id in liked_ids if movie_id in movie_to_row]

    if not liked_rows:
        return pd.DataFrame(columns=["movie_id", "similarity_score"])

    # Average the TF-IDF vectors of the user's liked movies
    user_profile = np.asarray(movie_features[liked_rows].mean(axis=0))

    # Compare the profile with every movie
    scores = cosine_similarity(user_profile, movie_features).ravel()

    recommendations = pd.DataFrame({"movie_id": movie_ids,"similarity_score": scores})

    # Exclude movies the user has already rated
    rated_ids = set(user_ratings["movie_id"])
    recommendations = recommendations[ ~recommendations["movie_id"].isin(rated_ids)]

    return recommendations.sort_values("similarity_score", ascending=False).head(n).reset_index(drop=True)

In [91]:
print(recommend_movies("Eva"))

[0.00886289 0.00786604 0.00844625 0.05636614 0.56727892 0.00740486
 0.00839135 0.0430748  0.59776118 0.00835739 0.59776118 0.00777684]
           movie_id  similarity_score
0          Downfall          0.056366
1         Malcolm x          0.043075
2  A Beautiful Mind          0.008863
